In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
from tqdm import tqdm
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import re
import numpy as np
import random
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
import time
import gc
from torch.utils.data import Sampler
from tqdm import tqdm
from timeit import default_timer as timer
import gc


In [2]:
#set TORCH_HOME=E:\MS Thesis\codes\DSTCT-master\code\weights
#os.environ['TORCH_HOME'] = r'E:\New_Project\CoronaryDominanceDataset\MultiTaskingLearning\weights'
os.environ['TORCH_HOME'] = r'E:\MS Thesis\codes\DSTCT-master\code\weights'

# MODEL

In [21]:
# Keep your existing XCAMultiTaskModel class unchanged
class XCAMultiTaskModel(nn.Module):
    def __init__(self, backbone, backbone_type, input_channels=1,
                 num_classes_occlusion=2, num_classes_frame_quality=2,
                 num_classes_dominance=2, hidden_dim=256,
                 sequential_model_type='lstm', sequential_hidden_dim=256,
                 num_sequential_layers=1, bidirectional=False,
                 num_heads_transformer=4, num_transformer_layers=2):

        super().__init__()
        self.backbone_type = backbone_type.lower()
        self.backbone = backbone
        self.input_channels = input_channels
        self.hidden_dim = hidden_dim

        # Parameters for the sequential model
        self.sequential_model_type = sequential_model_type.lower()
        self.sequential_hidden_dim = sequential_hidden_dim
        self.num_sequential_layers = num_sequential_layers
        self.bidirectional = bidirectional
        self.num_heads_transformer = num_heads_transformer
        self.num_transformer_layers = num_transformer_layers

        # 1. MODIFY FIRST CONV LAYER FOR GRAYSCALE INPUT
        self._modify_first_conv_layer()

        # 2. DEFINE FEATURE EXTRACTOR AND DETERMINE FEATURE DIMENSION
        self._setup_feature_extractor()

        # Shared spatial pooling
        self.spatial_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()

        # Define the sequential model for occlusion
        self._setup_sequential_model()

        # --- Task-specific heads with shared components ---
        # Shared feature projection to reduce computation
        self.shared_projection = nn.Sequential(
            nn.Linear(self.feature_dim, self.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        # Occlusion head
        self.occlusion_head = nn.Sequential(
            nn.Linear(self.temporal_feature_dim, self.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(self.hidden_dim, num_classes_occlusion)
        )

        # Frame Quality Head (Per-Frame) - uses shared projection
        self.frame_quality_head = nn.Linear(self.hidden_dim, num_classes_frame_quality)
        """self.frame_quality_head = nn.Sequential(
            nn.Linear(self.hidden_dim,self.hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(self.hidden_dim // 2, num_classes_frame_quality)
        )"""

        # Dominance Head (Per-Frame) - uses shared projection
        self.dominance_head = nn.Linear(self.hidden_dim, num_classes_dominance)
        """self.dominance_head = nn.Sequential(
            nn.Linear(self.hidden_dim,self.hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(self.hidden_dim // 2, num_classes_dominance)
        )"""

        # Add priority training components
        self.task_names = ['occlusion', 'frame_quality', 'dominance']
        self.num_tasks = len(self.task_names)
        
        # Initialize connection strength tracking
        self.connection_strengths = {task: [] for task in self.task_names}
        self.channel_groups = {}
        self._initialize_channel_groups()

    def _initialize_channel_groups(self):
        """Initialize empty channel groups for each task"""
        for task in self.task_names:
            self.channel_groups[task] = set()

    def _setup_feature_extractor(self):
        """Optimized feature extractor setup"""
        if hasattr(self.backbone, 'features'):
            self.extractor = self.backbone.features
            if "densenet" in self.backbone_type:
                self.feature_dim = self.backbone.classifier.in_features
            elif "efficientnet" in self.backbone_type or "mobilenet" in self.backbone_type:
                # More efficient feature dim detection
                if hasattr(self.backbone.classifier, 'in_features'):
                    self.feature_dim = self.backbone.classifier.in_features
                else:
                    # Use dummy input only if necessary
                    with torch.no_grad():
                        dummy_input = torch.randn(1, self.input_channels, 224, 224)
                        dummy_output = self.extractor(dummy_input)
                        self.feature_dim = dummy_output.shape[1]
        elif "resnet" in self.backbone_type:
            self.extractor = nn.Sequential(*list(self.backbone.children())[:-2])
            self.feature_dim = self.backbone.fc.in_features
        else:
            raise ValueError(f"Unsupported backbone type: {self.backbone_type}")

    def _setup_sequential_model(self):
        """Setup sequential model for temporal processing"""
        if self.sequential_model_type == 'lstm':
            self.occlusion_temporal_model = nn.LSTM(
                input_size=self.feature_dim,
                hidden_size=self.sequential_hidden_dim,
                num_layers=self.num_sequential_layers,
                batch_first=True,
                bidirectional=self.bidirectional
            )
        elif self.sequential_model_type == 'gru':
            self.occlusion_temporal_model = nn.GRU(
                input_size=self.feature_dim,
                hidden_size=self.sequential_hidden_dim,
                num_layers=self.num_sequential_layers,
                batch_first=True,
                bidirectional=self.bidirectional
            )
        else:
            raise ValueError(f"Unsupported sequential_model_type: {self.sequential_model_type}")

        # Output feature dimension from the sequential model
        self.temporal_feature_dim = self.sequential_hidden_dim * (2 if self.bidirectional else 1)

    def forward(self, images_dict):
        outputs = {}

        # --- Process Occlusion Images (Temporal) ---
        occlusion_clips = images_dict['occlusion_images']
        batch_size, seq_length, C, H, W = occlusion_clips.shape

        # Reshape for 2D CNN backbone
        occlusion_flat = occlusion_clips.view(batch_size * seq_length, C, H, W)

        # Extract spatial features
        occlusion_features_spatial_flat = self.extractor(occlusion_flat)

        # Apply spatial pooling and flattening
        occlusion_features_pooled_flat = self.flatten(
            self.spatial_pooling(occlusion_features_spatial_flat)
        )

        # Reshape for temporal processing
        occlusion_features_temporal_input = occlusion_features_pooled_flat.view(
            batch_size, seq_length, -1
        )

        # Pass through sequential model
        occlusion_temporal_output, _ = self.occlusion_temporal_model(occlusion_features_temporal_input)

        # Apply occlusion head
        occlusion_logits = self.occlusion_head(
            occlusion_temporal_output.contiguous().view(batch_size * seq_length, -1)
        )
        outputs['occlusion'] = occlusion_logits.view(batch_size, seq_length, -1)

        # --- Process Frame Quality Images ---
        fq_images = images_dict['frame_quality_images'].squeeze(1)
        fq_features = self.extractor(fq_images)
        fq_pooled = self.flatten(self.spatial_pooling(fq_features))
        fq_projected = self.shared_projection(fq_pooled)
        fq_logits = self.frame_quality_head(fq_projected)
        outputs['frame_quality'] = fq_logits.unsqueeze(1)

        # --- Process Dominance Images ---
        dom_images = images_dict['dominance_images'].squeeze(1)
        dom_features = self.extractor(dom_images)
        dom_pooled = self.flatten(self.spatial_pooling(dom_features))
        dom_projected = self.shared_projection(dom_pooled)
        dom_logits = self.dominance_head(dom_projected)
        outputs['dominance'] = dom_logits.unsqueeze(1)

        # Store shared features for priority computation
        outputs['shared_features'] = fq_projected  # Use frame quality shared features as reference

        return outputs

    def compute_connection_strength(self, task_name, gradients):
        """
        Compute connection strength for a given task based on gradients
        Uses a more stable computation method
        """
        if gradients is None:
            return 0.0

        # Compute L2 norm of gradients as connection strength with numerical stability
        strength = torch.norm(gradients.flatten(), p=2).item()

        # Store connection strength for tracking
        if task_name not in self.connection_strengths:
            self.connection_strengths[task_name] = []
        self.connection_strengths[task_name].append(strength)

        return strength

    def update_channel_groups(self, priority_task, gradients_dict):
        """Update channel groups based on priority task (Algorithm Step 9)"""
        if priority_task not in gradients_dict:
            return

        priority_gradients = gradients_dict[priority_task]
        if priority_gradients is None:
            return

        # Add channels with highest gradients to priority task's channel group
        # For 2D gradients (like Linear layer weights), we compute norm across one dimension
        if priority_gradients.dim() == 2:
            grad_norm = torch.norm(priority_gradients, dim=1)  # Norm across input features
        else:
            grad_norm = torch.norm(priority_gradients, dim=0)  # For 1D gradients

        # Select top channels based on gradient magnitude
        k = min(10, len(grad_norm))
        if k > 0:
            top_channels = torch.topk(grad_norm, k=k, largest=True).indices
            self.channel_groups[priority_task].update(top_channels.cpu().numpy())

    def project_gradients(self, gradients_dict, priority_task):
        """
        Project gradients based on channel groups and priorities with improved stability
        """
        if priority_task not in gradients_dict:
            return gradients_dict

        priority_gradients = gradients_dict[priority_task]
        if priority_gradients is None:
            return gradients_dict

        projected_gradients = {}

        for task_name, gradients in gradients_dict.items():
            if gradients is None:
                projected_gradients[task_name] = None
                continue

            if task_name == priority_task:
                # Priority task gradients remain unchanged
                projected_gradients[task_name] = gradients
            else:
                # Project other task gradients to reduce conflict with priority task
                projected_grad = gradients.clone()

                # Compute conflict with priority task
                flattened_grad = projected_grad.flatten()
                flattened_priority = priority_gradients.flatten()

                dot_product = torch.dot(flattened_grad, flattened_priority)

                if dot_product < 0:  # Conflicting gradients
                    # Project away from conflicting direction
                    priority_norm_sq = torch.dot(flattened_priority, flattened_priority)
                    if priority_norm_sq > 1e-8:
                        projection_coefficient = dot_product / priority_norm_sq
                        projection = projection_coefficient * flattened_priority
                        projected_flat = flattened_grad - projection
                        projected_grad = projected_flat.reshape(gradients.shape)

                projected_gradients[task_name] = projected_grad

        return projected_gradients

    def _modify_first_conv_layer(self):
        """Modifies the first convolutional layer for grayscale input"""
        if self.input_channels == 1:
            if "resnet" in self.backbone_type:
                old_conv = self.backbone.conv1
                self.backbone.conv1 = nn.Conv2d(
                    1, old_conv.out_channels, old_conv.kernel_size,
                    old_conv.stride, old_conv.padding, bias=old_conv.bias is not None
                )
            elif "densenet" in self.backbone_type:
                old_conv = self.backbone.features.conv0
                self.backbone.features.conv0 = nn.Conv2d(
                    1, old_conv.out_channels, old_conv.kernel_size,
                    old_conv.stride, old_conv.padding, bias=old_conv.bias is not None
                )
            elif "efficientnet" in self.backbone_type or "mobilenet" in self.backbone_type:
                old_conv = self.backbone.features[0][0]
                self.backbone.features[0][0] = nn.Conv2d(1, old_conv.out_channels, old_conv.kernel_size, old_conv.stride, old_conv.padding, bias=old_conv.bias)

    def freeze_backbone(self):
        """Freezes all parameters in the backbone."""
        for param in self.backbone.parameters():
            param.requires_grad = False

    def set_head_trainability(self, task_name, requires_grad, verbose=True):
        """Set trainability for specific task heads"""
        if task_name == 'occlusion':
            for param in self.occlusion_temporal_model.parameters():
                param.requires_grad = requires_grad
            for param in self.occlusion_head.parameters():
                param.requires_grad = requires_grad
        else:
            head = getattr(self, f"{task_name}_head", None)
            if head is not None:
                for param in head.parameters():
                    param.requires_grad = requires_grad

        if verbose:
            status = "TRAINABLE" if requires_grad else "FROZEN"
            print(f"Parameters for '{task_name}' head set to: {status}")


# DATALOADER

In [4]:
class OptimizedDataset(Dataset):
    """Optimized dataset with improved data loading"""
    def __init__(self, root_dir, split_type, view_type, clip_length, transform=None, fold_num=1):
        self.root_dir = root_dir
        self.split_type = split_type
        self.fold_num = fold_num
        self.transform = transform
        self.view_type = view_type
        self.clip_length = clip_length

        # Label mappings
        self.occlusion_label_map = {'nonoccluded': 0, 'occluded': 1}
        self.frame_quality_label_map = {'noninformative': 0, 'informative': 1}
        self.dominance_label_map = {'rightdom': 0, 'leftdom': 1}

        # Filename pattern
        self.filename_pattern = re.compile(r'(.+)_frame_(\d+)\.png')

        # Data structures
        self.occlusion_video_data_map = {}
        self.frame_quality_frame_data_list = []
        self.dominance_frame_data_list = []
        self.occlusion_clips_to_load = []

        # Pre-computed indices for faster random sampling
        self._fq_indices = None
        self._dom_indices = None

        self._load_and_index_data()
        self._prepare_random_indices()

    def _prepare_random_indices(self):
        """Pre-compute random indices for faster sampling"""
        self._fq_indices = np.arange(len(self.frame_quality_frame_data_list))
        self._dom_indices = np.arange(len(self.dominance_frame_data_list))
        np.random.shuffle(self._fq_indices)
        np.random.shuffle(self._dom_indices)

    def _load_and_index_data(self):
        """Load and index data with optimizations"""
        task_configs = {
            'occlusion': {
                'csv_prefix': 'occlusion',
                'label_map': self.occlusion_label_map,
                'image_base_path': os.path.join(self.root_dir, 'occlusion', 'DATA_RCA'),
            },
            'framequality': {
                'csv_prefix': 'framequality',
                'label_map': self.frame_quality_label_map,
                'image_base_path': os.path.join(self.root_dir, 'framequality', f'DATA_{self.view_type}'),
            },
            'dominance': {
                'csv_prefix': 'dom',
                'label_map': self.dominance_label_map,
                'image_base_path': os.path.join(self.root_dir, 'dominance', f'DATA_{self.view_type}'),
            }
        }

        # Process occlusion data
        occlusion_config = task_configs['occlusion']
        csv_path = os.path.join(
            occlusion_config['image_base_path'], 'labels',
            f"{occlusion_config['csv_prefix']}_{self.split_type}_labels_fold_{self.fold_num}.csv"
        )

        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            for _, row in df.iterrows():
                filename, label_str = str(row['filename']), str(row['label']).strip().lower()
                match = self.filename_pattern.match(filename)
                if not match:
                    continue

                video_id, frame_num = match.group(1), int(match.group(2))
                if video_id not in self.occlusion_video_data_map:
                    self.occlusion_video_data_map[video_id] = {}

                label_value = occlusion_config['label_map'].get(label_str)
                if label_value is None:
                    continue

                image_path = os.path.join(occlusion_config['image_base_path'], label_str, filename)
                if os.path.exists(image_path):
                    self.occlusion_video_data_map[video_id][frame_num] = {
                        'path': image_path, 'label': label_value
                    }

        # Generate clips more efficiently
        for video_id, frames_dict in self.occlusion_video_data_map.items():
            sorted_frames = sorted(frames_dict.keys())
            if len(sorted_frames) >= self.clip_length:
                for i in range(len(sorted_frames) - self.clip_length + 1):
                    self.occlusion_clips_to_load.append({
                        'video_id': video_id,
                        'clip_original_frame_indices': sorted_frames[i:i + self.clip_length]
                    })

        # Process frame quality and dominance data
        for task_name in ['framequality', 'dominance']:
            config = task_configs[task_name]
            csv_path = os.path.join(
                config['image_base_path'], 'labels',
                f"{config['csv_prefix']}_{self.split_type}_labels_fold_{self.fold_num}.csv"
            )

            target_list = (self.frame_quality_frame_data_list if task_name == 'framequality'
                          else self.dominance_frame_data_list)

            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                for _, row in df.iterrows():
                    filename, label_str = str(row['filename']), str(row['label']).strip().lower()
                    label_value = config['label_map'].get(label_str)
                    if label_value is None:
                        continue

                    image_path = os.path.join(config['image_base_path'], label_str, filename)
                    if os.path.exists(image_path):
                        target_list.append({'path': image_path, 'label': label_value})

    def __len__(self):
        return len(self.occlusion_clips_to_load)

    def __getitem__(self, idx):
        # Get occlusion clip
        clip_info = self.occlusion_clips_to_load[idx]
        video_id = clip_info['video_id']
        frame_indices = clip_info['clip_original_frame_indices']

        occlusion_frames = []
        occlusion_labels = []

        for frame_idx in frame_indices:
            frame_data = self.occlusion_video_data_map[video_id][frame_idx]
            image = Image.open(frame_data['path'])
            occlusion_frames.append(image)
            occlusion_labels.append(frame_data['label'])

        if self.transform:
            occlusion_frames_tensor = self.transform(occlusion_frames)
        else:
            occlusion_frames_tensor = torch.stack([transforms.ToTensor()(f) for f in occlusion_frames])

        # Optimized random sampling using pre-computed indices
        fq_idx = self._fq_indices[idx % len(self._fq_indices)]
        dom_idx = self._dom_indices[idx % len(self._dom_indices)]

        fq_entry = self.frame_quality_frame_data_list[fq_idx]
        dom_entry = self.dominance_frame_data_list[dom_idx]

        # Load and transform single frame images
        fq_image = Image.open(fq_entry['path'])
        dom_image = Image.open(dom_entry['path'])

        if self.transform:
            fq_image_tensor = self.transform.transform(fq_image)
            dom_image_tensor = self.transform.transform(dom_image)
        else:
            fq_image_tensor = transforms.ToTensor()(fq_image)
            dom_image_tensor = transforms.ToTensor()(dom_image)

        return {
            "occlusion_images": occlusion_frames_tensor,
            "frame_quality_images": fq_image_tensor.unsqueeze(0),
            "dominance_images": dom_image_tensor.unsqueeze(0),
        }, {
            "occlusion_labels": torch.tensor(occlusion_labels, dtype=torch.long),
            "frame_quality_labels": torch.tensor([fq_entry['label']], dtype=torch.long),
            "dominance_labels": torch.tensor([dom_entry['label']], dtype=torch.long),
        }

class OptimizedClipTransform:
    """Optimized transform with reduced operations"""
    def __init__(self, img_size, mean, std, apply_augmentations=True):
        # Base transforms
        base_transforms = [
            transforms.Resize(img_size, antialias=True),  # Add antialias for better quality
            transforms.ToTensor(),
        ]

        # Only add augmentations if requested and for training
        if apply_augmentations:
            augment_transforms = [
                transforms.RandomHorizontalFlip(p=0.3),  # Reduced probability
                transforms.RandomRotation(5),  # Reduced rotation range
            ]
            # Insert augmentations before ToTensor
            base_transforms = base_transforms[:-1] + augment_transforms + base_transforms[-1:]

        base_transforms.append(transforms.Normalize(mean=mean, std=std))
        self.transform = transforms.Compose(base_transforms)

    def __call__(self, images):
        # Use list comprehension for efficiency
        transformed_images = [self.transform(img) for img in images]
        return torch.stack(transformed_images)


# TRAINING SECTION

## TRAINING FUNCTIONS

In [5]:
def calculate_accuracy(logits, labels, ignore_label_value):
    """
    Calculates accuracy, ignoring specified label values.
    Assumes logits are (Batch, NumClasses) and labels are (Batch).
    """
    valid_mask = (labels != ignore_label_value)
    if valid_mask.sum() == 0:
        return 0.0, 0

    _, predicted = torch.max(logits, 1)
    correct = (predicted[valid_mask] == labels[valid_mask]).sum().item()
    total_valid = valid_mask.sum().item()
    return correct / total_valid if total_valid > 0 else 0.0, total_valid


In [6]:
def _apply_two_phase_mtl_algorithm(model, losses, active_tasks, optimizer, valid_task_losses, 
                                  current_epoch, total_epochs, is_phase_1=True):
    """
    Proper implementation of Algorithm 1: Connection Strength-based Optimization for Multi-task Learning
    
    Args:
        model: The multi-task model
        losses: Dictionary of task losses
        active_tasks: List of active tasks
        optimizer: Optimizer
        valid_task_losses: List of valid task losses for fallback
        current_epoch: Current epoch number (0-indexed)
        total_epochs: Total number of epochs
        is_phase_1: Whether we're in phase 1 (learning priority) or phase 2 (conserving priority)
    """
    K = len(active_tasks)  # Number of tasks
    E = total_epochs
    e = current_epoch
    
    if is_phase_1:
        # Phase 1: Learning the task priority
        # Step 1: Randomly choose P ~ U(0,1)
        P = random.random()
        
        # Step 2: if P ≥ e/E then
        if P >= e / E:
            # Steps 3-4: Simple multi-task learning (update gradients one-by-one)
            optimizer.zero_grad()
            total_loss = sum(valid_task_losses)
            total_loss.backward()
            optimizer.step()
            return total_loss.item()
        else:
            # Continue to Phase 2 logic even in Phase 1 when P < e/E
            pass
    
    # Phase 2: Conserving the task priority (or Phase 1 when P < e/E)
    
    # Step 6: Initialize all CG_v as empty set {} in the shared convolutional layer
    # This is done once at the beginning, so we skip if already initialized
    if not hasattr(model, 'channel_groups') or not model.channel_groups:
        model.channel_groups = {task: set() for task in active_tasks}
    
    # Steps 7-9: Determine priority task and update channel groups
    connection_strengths = {}
    task_gradients = {}
    
    # Compute connection strengths (S_i^τ) for each task
    for task in active_tasks:
        if task in losses:
            optimizer.zero_grad()
            losses[task].backward(retain_graph=True)
            
            # Collect gradients from shared convolutional layer
            # For your model, this would be from the backbone feature extractor
            shared_gradients = []
            for name, param in model.named_parameters():
                if 'extractor' in name and param.grad is not None:  # Shared conv layers
                    shared_gradients.append(param.grad.flatten())
            
            if shared_gradients:
                task_gradient = torch.cat(shared_gradients)
                task_gradients[task] = task_gradient
                # Connection strength is the L2 norm of gradients
                connection_strengths[task] = torch.norm(task_gradient, p=2).item()
            else:
                task_gradients[task] = None
                connection_strengths[task] = 0.0
            
            optimizer.zero_grad()
    
    # Step 8: ν = argmax_i S_i^τ (determine the top priority task ν)
    if connection_strengths:
        priority_task = max(connection_strengths.keys(), key=lambda k: connection_strengths[k])
        
        # Step 9: CG_ν = CG_ν + {c_p^out} (classify channel with task ν)
        if priority_task in task_gradients and task_gradients[priority_task] is not None:
            # Add top gradient channels to priority task's channel group
            grad_norms = torch.abs(task_gradients[priority_task])
            top_k = min(10, len(grad_norms))  # Add top 10 channels
            if top_k > 0:
                top_channels = torch.topk(grad_norms, k=top_k).indices
                model.channel_groups[priority_task].update(top_channels.cpu().numpy().tolist())
        
        # Steps 10-14: Gradient projection for conflict resolution
        optimizer.zero_grad()
        total_loss = sum(valid_task_losses)
        total_loss.backward()
        
        # Apply gradient projection
        _apply_gradient_projection(model, task_gradients, priority_task, active_tasks)
        
        optimizer.step()
        return total_loss.item()
    
    # Fallback: simple multi-task learning
    optimizer.zero_grad()
    total_loss = sum(valid_task_losses)
    total_loss.backward()
    optimizer.step()
    return total_loss.item()


def _apply_gradient_projection(model, task_gradients, priority_task, active_tasks):
    """
    Apply gradient projection according to Algorithm 1 steps 10-14
    """
    priority_grad = task_gradients.get(priority_task)
    if priority_grad is None:
        return
    
    # Step 11: Let {G_i,1, ..., G_i,K} are gradients of CG_i
    # Step 12-14: Gradient projection for non-priority tasks
    
    with torch.no_grad():
        for task in active_tasks:
            if task != priority_task and task in task_gradients and task_gradients[task] is not None:
                task_grad = task_gradients[task]
                
                # Only project if gradients have compatible dimensions
                if task_grad.shape == priority_grad.shape:
                    # Step 13: if G_i,j · G_i,ν < 0 then (conflict detection)
                    dot_product = torch.dot(task_grad, priority_grad)
                    
                    if dot_product < 0:  # Conflicting gradients
                        # Step 14: G_i,j = G_i,j - (G_i,j·G_i,ν)/(||G_i,ν||^2) · G_i,ν
                        priority_norm_sq = torch.dot(priority_grad, priority_grad)
                        if priority_norm_sq > 1e-8:
                            projection_coeff = dot_product / priority_norm_sq
                            projected_grad = task_grad - projection_coeff * priority_grad
                            
                            # Apply projected gradients back to the model
                            _apply_projected_gradients_to_model(model, projected_grad, task)


def _apply_projected_gradients_to_model(model, projected_grad, task):
    """
    Apply projected gradients back to the appropriate model parameters
    """
    # Apply to shared convolutional layers (extractor)
    start_idx = 0
    for name, param in model.named_parameters():
        if 'extractor' in name and param.grad is not None:
            param_size = param.grad.numel()
            if start_idx + param_size <= len(projected_grad):
                param.grad.data = projected_grad[start_idx:start_idx + param_size].reshape(param.grad.shape)
                start_idx += param_size

In [7]:
def train_two_phase_priority_temporal_mtl(model, train_dl, val_dl,
                                        criterion_occlusion, criterion_frame_quality, criterion_dominance,
                                        optimizer, num_epochs, device, checkpoint_name, scheduler,
                                        accumulation_steps=4, patience=10, fold_idx=1,
                                        ignore_label_value=-1, resume=False,
                                        current_view_type='RCA', save_freq=10,
                                        phase_transition_epoch=None):
    """Two-phase priority training for temporal MTL with comprehensive logging"""
    
    # Use mixed precision training
    scaler = torch.amp.GradScaler('cuda')
    model.to(device)

    # Enable optimizations
    torch.backends.cudnn.benchmark = True

    start_epoch = 0
    best_val_accuracy = 0.0
    training_history = []
    epochs_no_improve = 0

    # Set phase transition point (default to halfway through training)
    if phase_transition_epoch is None:
        phase_transition_epoch = num_epochs // 2

    print(f"Two-Phase Temporal MTL Training Setup:")
    print(f"Phase 1 (Learning Priority): Epochs 1-{phase_transition_epoch}")
    print(f"Phase 2 (Conserving Priority): Epochs {phase_transition_epoch+1}-{num_epochs}")

    # Create checkpoint directory structure
    checkpoint_dir = os.path.dirname(checkpoint_name)
    best_model_path = checkpoint_name
    latest_checkpoint_path = os.path.join(checkpoint_dir, 'latest_checkpoint.pth')

    # Create periodic checkpoint directory if save_freq > 0
    if save_freq > 0:
        periodic_checkpoint_dir = os.path.join(checkpoint_dir, 'periodic_checkpoints')
        os.makedirs(periodic_checkpoint_dir, exist_ok=True)

    print(f"Checkpoint directory: {checkpoint_dir}")
    print(f"Best model will be saved as: {best_model_path}")
    print(f"Latest checkpoint will be saved as: {latest_checkpoint_path}")
    if save_freq > 0:
        print(f"Periodic checkpoints will be saved every {save_freq} epochs in: {periodic_checkpoint_dir}")

    # Configure head trainability based on view type
    if current_view_type == 'LCA':
        print("\n--- Configuring model for LCA training (Occlusion head frozen) ---")
        model.set_head_trainability('occlusion', False, verbose=True)
        model.set_head_trainability('frame_quality', True, verbose=True)
        model.set_head_trainability('dominance', True, verbose=True)
        active_tasks = ['frame_quality', 'dominance']
    elif current_view_type == 'RCA':
        print("\n--- Configuring model for RCA training (All heads trainable) ---")
        model.set_head_trainability('occlusion', True, verbose=True)
        model.set_head_trainability('frame_quality', True, verbose=True)
        model.set_head_trainability('dominance', True, verbose=True)
        active_tasks = ['occlusion', 'frame_quality', 'dominance']
    else:
        print(f"Warning: Unknown current_view_type '{current_view_type}'. All heads will be trainable.")
        active_tasks = ['occlusion', 'frame_quality', 'dominance']

    # Resume logic - check for latest checkpoint first, then best model
    resume_path = None
    if resume:
        if os.path.exists(latest_checkpoint_path):
            resume_path = latest_checkpoint_path
            print(f"Found latest checkpoint: {latest_checkpoint_path}")
        elif os.path.exists(best_model_path):
            resume_path = best_model_path
            print(f"Found best model checkpoint: {best_model_path}")

    if resume_path:
        print(f"Resuming training from: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=device, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if 'scaler_state_dict' in checkpoint:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint.get('epoch', -1) + 1
        best_val_accuracy = checkpoint.get('best_val_accuracy', 0.0)
        training_history = checkpoint.get('history', [])
        # Restore priority training state
        if 'channel_groups' in checkpoint:
            model.channel_groups = checkpoint['channel_groups']
        if 'connection_strengths' in checkpoint:
            model.connection_strengths = checkpoint['connection_strengths']
        print(f"Resumed from Epoch {start_epoch}. Best accuracy: {best_val_accuracy:.4f}")
    else:
        if resume:
            print("No checkpoint found. Starting training from scratch.")
        else:
            print("Starting training from scratch.")

    # Pre-allocate tensors for better memory management
    device_kwargs = {'device': device, 'non_blocking': True}

    for epoch in range(start_epoch, num_epochs):
        epoch_start_time = time.time()
        
        # Determine current phase
        is_phase_1 = epoch < phase_transition_epoch
        phase_name = "Learning Priority" if is_phase_1 else "Conserving Priority"

        print(f"\nEpoch {epoch+1}/{num_epochs} - Phase {'1' if is_phase_1 else '2'}: {phase_name}")

        # Training phase
        model.train()
        train_metrics = {
            'loss': 0.0,
            'occlusion_loss': 0.0, 'frame_quality_loss': 0.0, 'dominance_loss': 0.0,
            'occlusion_correct': 0, 'frame_quality_correct': 0, 'dominance_correct': 0,
            'occlusion_total': 0, 'frame_quality_total': 0, 'dominance_total': 0
        }

        optimizer.zero_grad()

        # Use tqdm with reduced update frequency for better performance
        train_pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs} [Train]",
                         leave=False, mininterval=1.0)

        for batch_idx, (inputs, labels) in enumerate(train_pbar):
            # Move data to device with non_blocking for better performance
            images_dict = {k: v.to(**device_kwargs) for k, v in inputs.items()}
            labels_dict = {k: v.to(**device_kwargs) for k, v in labels.items()}

            with torch.amp.autocast("cuda"):
                outputs = model(images_dict)

                # Compute losses more efficiently
                losses = {}

                # Occlusion loss
                occlusion_logits = outputs['occlusion'].view(-1, outputs['occlusion'].shape[-1])
                occlusion_labels = labels_dict['occlusion_labels'].view(-1)
                valid_occ_mask = occlusion_labels != ignore_label_value
                if valid_occ_mask.any():
                    losses['occlusion'] = criterion_occlusion(
                        occlusion_logits[valid_occ_mask], occlusion_labels[valid_occ_mask]
                    )
                else:
                    losses['occlusion'] = torch.tensor(0.0, device=device)

                # Frame quality loss
                fq_logits = outputs['frame_quality'].squeeze(1)
                fq_labels = labels_dict['frame_quality_labels'].squeeze(1)
                valid_fq_mask = fq_labels != ignore_label_value
                if valid_fq_mask.any():
                    losses['frame_quality'] = criterion_frame_quality(
                        fq_logits[valid_fq_mask], fq_labels[valid_fq_mask]
                    )
                else:
                    losses['frame_quality'] = torch.tensor(0.0, device=device)

                # Dominance loss
                dom_logits = outputs['dominance'].squeeze(1)
                dom_labels = labels_dict['dominance_labels'].squeeze(1)
                valid_dom_mask = dom_labels != ignore_label_value
                if valid_dom_mask.any():
                    losses['dominance'] = criterion_dominance(
                        dom_logits[valid_dom_mask], dom_labels[valid_dom_mask]
                    )
                else:
                    losses['dominance'] = torch.tensor(0.0, device=device)

                # Calculate losses only for active tasks with valid samples
                valid_task_losses = []
                for task in active_tasks:
                    if task in losses and losses[task].item() > 0:
                        valid_task_losses.append(losses[task])

                if len(valid_task_losses) == 0:
                    continue  # Skip if no valid tasks
                    
            # Apply the proper two-phase MTL algorithm
            batch_loss = _apply_two_phase_mtl_algorithm(
                model=model,
                losses=losses,
                active_tasks=active_tasks,
                optimizer=optimizer,
                valid_task_losses=valid_task_losses,
                current_epoch=epoch,
                total_epochs=num_epochs,
                is_phase_1=is_phase_1
            )

            train_metrics['loss'] += batch_loss

            # Accumulate individual task losses for logging
            for task in active_tasks:
                if task in losses:
                    train_metrics[f'{task}_loss'] += losses[task].item()

            # Calculate accuracies
            if current_view_type == 'RCA' and valid_occ_mask.any():
                _, occ_preds = torch.max(occlusion_logits[valid_occ_mask], 1)
                train_metrics['occlusion_correct'] += (occ_preds == occlusion_labels[valid_occ_mask]).sum().item()
                train_metrics['occlusion_total'] += valid_occ_mask.sum().item()

            if valid_fq_mask.any():
                _, fq_preds = torch.max(fq_logits[valid_fq_mask], 1)
                train_metrics['frame_quality_correct'] += (fq_preds == fq_labels[valid_fq_mask]).sum().item()
                train_metrics['frame_quality_total'] += valid_fq_mask.sum().item()

            if valid_dom_mask.any():
                _, dom_preds = torch.max(dom_logits[valid_dom_mask], 1)
                train_metrics['dominance_correct'] += (dom_preds == dom_labels[valid_dom_mask]).sum().item()
                train_metrics['dominance_total'] += valid_dom_mask.sum().item()

        # Validation phase
        model.eval()
        val_metrics = {
            'loss': 0.0,
            'occlusion_loss': 0.0, 'frame_quality_loss': 0.0, 'dominance_loss': 0.0,
            'occlusion_correct': 0, 'frame_quality_correct': 0, 'dominance_correct': 0,
            'occlusion_total': 0, 'frame_quality_total': 0, 'dominance_total': 0
        }

        with torch.no_grad():
            val_pbar = tqdm(val_dl, desc=f"Epoch {epoch+1}/{num_epochs} [Val]",
                           leave=False, mininterval=1.0)

            for inputs, labels in val_pbar:
                images_dict = {k: v.to(**device_kwargs) for k, v in inputs.items()}
                labels_dict = {k: v.to(**device_kwargs) for k, v in labels.items()}

                with torch.amp.autocast("cuda"):
                    outputs = model(images_dict)

                    # Validation loss computation
                    losses = {}

                    occlusion_logits = outputs['occlusion'].view(-1, outputs['occlusion'].shape[-1])
                    occlusion_labels = labels_dict['occlusion_labels'].view(-1)
                    valid_occ_mask = occlusion_labels != ignore_label_value
                    losses['occlusion'] = (criterion_occlusion(occlusion_logits[valid_occ_mask],
                                                             occlusion_labels[valid_occ_mask])
                                         if valid_occ_mask.any() else torch.tensor(0.0, device=device))

                    fq_logits = outputs['frame_quality'].squeeze(1)
                    fq_labels = labels_dict['frame_quality_labels'].squeeze(1)
                    valid_fq_mask = fq_labels != ignore_label_value
                    losses['frame_quality'] = (criterion_frame_quality(fq_logits[valid_fq_mask],
                                                                      fq_labels[valid_fq_mask])
                                              if valid_fq_mask.any() else torch.tensor(0.0, device=device))

                    dom_logits = outputs['dominance'].squeeze(1)
                    dom_labels = labels_dict['dominance_labels'].squeeze(1)
                    valid_dom_mask = dom_labels != ignore_label_value
                    losses['dominance'] = (criterion_dominance(dom_logits[valid_dom_mask],
                                                             dom_labels[valid_dom_mask])
                                         if valid_dom_mask.any() else torch.tensor(0.0, device=device))

                # Update validation metrics
                val_metrics['loss'] += sum(losses.values()).item()
                for task in ['occlusion', 'frame_quality', 'dominance']:
                    val_metrics[f'{task}_loss'] += losses[task].item()

                # Validation accuracies
                if current_view_type == 'RCA' and valid_occ_mask.any():
                    _, occ_preds = torch.max(occlusion_logits[valid_occ_mask], 1)
                    val_metrics['occlusion_correct'] += (occ_preds == occlusion_labels[valid_occ_mask]).sum().item()
                    val_metrics['occlusion_total'] += valid_occ_mask.sum().item()

                if valid_fq_mask.any():
                    _, fq_preds = torch.max(fq_logits[valid_fq_mask], 1)
                    val_metrics['frame_quality_correct'] += (fq_preds == fq_labels[valid_fq_mask]).sum().item()
                    val_metrics['frame_quality_total'] += valid_fq_mask.sum().item()

                if valid_dom_mask.any():
                    _, dom_preds = torch.max(dom_logits[valid_dom_mask], 1)
                    val_metrics['dominance_correct'] += (dom_preds == dom_labels[valid_dom_mask]).sum().item()
                    val_metrics['dominance_total'] += valid_dom_mask.sum().item()

        # Calculate epoch metrics
        epoch_duration = time.time() - epoch_start_time
        num_batches_train = len(train_dl)
        num_batches_val = len(val_dl)

        # Training metrics
        avg_train_loss = train_metrics['loss'] / num_batches_train
        avg_val_loss = val_metrics['loss'] / num_batches_val

        # Calculate accuracies
        train_accuracies = []
        val_accuracies = []

        history = {
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'phase': 1 if is_phase_1 else 2,
        }

        for task in ['occlusion', 'frame_quality', 'dominance']:
            # Training
            train_acc = (train_metrics[f'{task}_correct'] / train_metrics[f'{task}_total']
                        if train_metrics[f'{task}_total'] > 0 else 0.0)
            val_acc = (val_metrics[f'{task}_correct'] / val_metrics[f'{task}_total']
                      if val_metrics[f'{task}_total'] > 0 else 0.0)

            history[f'train_{task}_loss'] = train_metrics[f'{task}_loss'] / num_batches_train
            history[f'val_{task}_loss'] = val_metrics[f'{task}_loss'] / num_batches_val
            history[f'train_{task}_acc'] = train_acc
            history[f'val_{task}_acc'] = val_acc

            if val_metrics[f'{task}_total'] > 0:
                val_accuracies.append(val_acc)
            if train_metrics[f'{task}_total'] > 0:
                train_accuracies.append(train_acc)

        # Overall accuracy
        avg_val_overall_accuracy = sum(val_accuracies) / len(val_accuracies) if val_accuracies else 0.0
        history['avg_val_overall_accuracy'] = avg_val_overall_accuracy
        training_history.append(history)

        # Print epoch summary
        print(f"Epoch {epoch+1}/{num_epochs} - Duration: {epoch_duration:.2f}s")
        print(f"Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")
        for task in ['occlusion', 'frame_quality', 'dominance']:
            print(f"  {task.title()}: Train Acc: {history[f'train_{task}_acc']:.4f}, "
                  f"Val Acc: {history[f'val_{task}_acc']:.4f}")
        print(f"  Overall Val Accuracy: {avg_val_overall_accuracy:.4f}")

        # Learning rate scheduling
        scheduler.step(avg_val_overall_accuracy)

        # Prepare checkpoint data
        checkpoint_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_accuracy': best_val_accuracy,
            'history': training_history,
            'current_val_accuracy': avg_val_overall_accuracy,
            'fold_idx': fold_idx,
            'view_type': current_view_type,
            'channel_groups': model.channel_groups,
            'connection_strengths': model.connection_strengths,
            'phase_transition_epoch': phase_transition_epoch,
        }

        # Save best model
        if avg_val_overall_accuracy > best_val_accuracy:
            best_val_accuracy = avg_val_overall_accuracy
            epochs_no_improve = 0

            # Update best accuracy in checkpoint data
            checkpoint_data['best_val_accuracy'] = best_val_accuracy

            torch.save(checkpoint_data, best_model_path)
            print(f"New best validation accuracy: {best_val_accuracy:.4f}. Best model saved.")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}. No improvement for {patience} epochs.")
                break

        # Always save latest checkpoint (overwrite previous)
        torch.save(checkpoint_data, latest_checkpoint_path)

        # Save periodic checkpoint if enabled
        if save_freq > 0 and (epoch + 1) % save_freq == 0:
            periodic_checkpoint_path = os.path.join(
                periodic_checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth'
            )
            torch.save(checkpoint_data, periodic_checkpoint_path)
            print(f"Periodic checkpoint saved: {periodic_checkpoint_path}")

        # Memory cleanup
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

    # Save last epoch model
    last_epoch_checkpoint_name = os.path.join(os.path.dirname(checkpoint_name), 
                                            os.path.basename(checkpoint_name).replace("_best.pth", "_last_epoch.pth"))
    torch.save(checkpoint_data, last_epoch_checkpoint_name)
    print(f"Saved last epoch model to {last_epoch_checkpoint_name} (Epoch {epoch+1})")

    return pd.DataFrame(training_history)

## NECESSARY FUNCTIONS

In [10]:
def create_optimized_model(backbone_type='densenet121', input_channels=1, hidden_dim=128):
    """Create an optimized model with better defaults"""
    if backbone_type == 'densenet121':
        backbone_model = models.densenet121(weights='DEFAULT', memory_efficient=True)
    elif backbone_type == 'resnet50':
        backbone_model = models.resnet50(weights='DEFAULT')
    elif backbone_type == 'resnet18':
        backbone_model = models.resnet18(weights='DEFAULT')
    elif backbone_type == 'mobilenet_v2':
        backbone_model = models.mobilenet_v2(weights='DEFAULT')
    elif backbone_type == 'efficientnet_b0':
        backbone_model = models.efficientnet_b0(weights='DEFAULT')
    else:
        raise ValueError(f"Unsupported backbone: {backbone_type}")

    model = XCAMultiTaskModel(
        backbone=backbone_model,
        backbone_type=backbone_type,
        input_channels=input_channels,
        num_classes_occlusion=2,
        num_classes_frame_quality=2,
        num_classes_dominance=2,
        hidden_dim=hidden_dim,
        sequential_model_type='lstm',
        sequential_hidden_dim=hidden_dim,
        num_sequential_layers=5,
        bidirectional=False
    )
    return model

def create_optimized_dataloaders(root_dir, view_type, clip_length, batch_size,
                                num_workers, fold_num=1, img_size=(512, 512)):
    """Create optimized dataloaders with better settings"""

    # Optimized transforms
    train_transform = OptimizedClipTransform(
        img_size=img_size,
        mean=[0.5485],
        std=[0.1407],
        apply_augmentations=True
    )

    val_transform = OptimizedClipTransform(
        img_size=img_size,
        mean=[0.5485],
        std=[0.1407],
        apply_augmentations=False
    )

    # Create datasets
    train_dataset = OptimizedDataset(
        root_dir=root_dir,
        fold_num=fold_num,
        split_type='train',
        view_type=view_type,
        clip_length=clip_length,
        transform=train_transform
    )

    val_dataset = OptimizedDataset(
        root_dir=root_dir,
        fold_num=fold_num,
        split_type='val',
        view_type=view_type,
        clip_length=clip_length,
        transform=val_transform
    )

    # Optimized dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
        prefetch_factor=2 if num_workers > 0 else None,
        drop_last=True  # For consistent batch sizes
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
        prefetch_factor=2 if num_workers > 0 else None
    )

    return train_loader, val_loader

def setup_optimized_training(model, learning_rate=1e-4):
    """Setup optimized training components"""

    # More efficient optimizer
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=1e-4,
        eps=1e-7
    )

    # Better scheduler
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='max',
        patience=3,
        factor=0.5,
        min_lr=1e-6
    )

    # Loss functions
    criterion_occlusion = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.1)
    criterion_frame_quality = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.1)
    criterion_dominance = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.1)

    return optimizer, scheduler, criterion_occlusion, criterion_frame_quality, criterion_dominance


## START TRAINING

In [9]:
ROOT_DATA_DIR = r'E:\Morshedul\CoronarDominance\MTL_DATASET'
VIEW_TYPE = 'LCA'
CLIP_LENGTH = 5
BATCH_SIZE = 8
NUM_WORKERS = 0
IMG_SIZE = (512, 512)
FOLD_NUM = 1
EPOCHS = 50
ACCUMULATION_STEPS = 2
PATIENCE = 100
IGNORE_LABEL_VALUE = -1
RESUME = False
BACKBONE = 'densenet121'
INPUT_CHANNELS = 1
HIDDEN_DIM = 128

In [10]:
# Create model and components
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_optimized_model(BACKBONE, input_channels=INPUT_CHANNELS, hidden_dim=HIDDEN_DIM)

rca_checkpoint_path = r'E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results\TwoPhasePriority_Densenet121_temporal_mtl_best.pth'
# Load the RCA checkpoint
checkpoint_rca = torch.load(rca_checkpoint_path, map_location=device,weights_only = False)
pretrained_dict = checkpoint_rca['model_state_dict']
# Filter out backbone and extractor weights (shared feature extraction)
shared_dict = {k: v for k, v in pretrained_dict.items() 
               if k.startswith('backbone.') or k.startswith('extractor.')}

# Update model
model_dict = model.state_dict()
model_dict.update(shared_dict)
model.load_state_dict(model_dict)

model.to(device)

# Create dataloaders
train_loader, val_loader = create_optimized_dataloaders(
    ROOT_DATA_DIR, VIEW_TYPE, CLIP_LENGTH, BATCH_SIZE, NUM_WORKERS, FOLD_NUM, IMG_SIZE
)

# Setup training
optimizer, scheduler, criterion_occ, criterion_fq, criterion_dom = setup_optimized_training(model)

# Training parameters
checkpoint_path = r"E:\Morshedul\CoronarDominance\MTL_results\LCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results"
os.makedirs(checkpoint_path, exist_ok=True)
checkpoint_name = os.path.join(checkpoint_path, 'TwoPhasePriority_Densenet121_temporal_mtl_best.pth')


In [11]:
print("Starting two-phase priority temporal multi-task training...")
training_history = train_two_phase_priority_temporal_mtl(
    model=model,
    train_dl=train_loader,
    val_dl=val_loader,
    criterion_occlusion=criterion_occ,
    criterion_frame_quality=criterion_fq,
    criterion_dominance=criterion_dom,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=EPOCHS,
    device=device,
    checkpoint_name=checkpoint_name,
    accumulation_steps=ACCUMULATION_STEPS,
    patience=PATIENCE,
    fold_idx=FOLD_NUM,
    ignore_label_value=IGNORE_LABEL_VALUE,
    resume=RESUME,
    current_view_type=VIEW_TYPE,
    phase_transition_epoch=EPOCHS // 2  # Transition at halfway point
)

print("Two-phase priority temporal MTL training completed!")

Starting two-phase priority temporal multi-task training...
Two-Phase Temporal MTL Training Setup:
Phase 1 (Learning Priority): Epochs 1-25
Phase 2 (Conserving Priority): Epochs 26-50
Checkpoint directory: E:\Morshedul\CoronarDominance\MTL_results\LCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results
Best model will be saved as: E:\Morshedul\CoronarDominance\MTL_results\LCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results\TwoPhasePriority_Densenet121_temporal_mtl_best.pth
Latest checkpoint will be saved as: E:\Morshedul\CoronarDominance\MTL_results\LCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results\latest_checkpoint.pth
Periodic checkpoints will be saved every 10 epochs in: E:\Morshedul\CoronarDominance\MTL_results\LCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results\periodic_checkpoints

--- Configuring model for LCA training (Occlusion head frozen) ---
Parameters for 'occlusion' head set to: FROZEN
Parameters for 'frame_quality' head set to: TRAINABLE
Paramet

Epoch 1/50 - Duration: 2203.76s
Train Loss: 0.6881 - Val Loss: 1.3619
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9631, Val Acc: 0.9819
  Dominance: Train Acc: 0.8737, Val Acc: 0.8801
  Overall Val Accuracy: 0.9310
New best validation accuracy: 0.9310. Best model saved.

Epoch 2/50 - Phase 1: Learning Priority


Epoch 2/50 - Duration: 1870.78s
Train Loss: 0.5596 - Val Loss: 1.3356
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9839, Val Acc: 0.9716
  Dominance: Train Acc: 0.9359, Val Acc: 0.8793
  Overall Val Accuracy: 0.9255

Epoch 3/50 - Phase 1: Learning Priority


Epoch 3/50 - Duration: 1867.58s
Train Loss: 0.5183 - Val Loss: 1.3765
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9882, Val Acc: 0.9797
  Dominance: Train Acc: 0.9528, Val Acc: 0.8546
  Overall Val Accuracy: 0.9171

Epoch 4/50 - Phase 1: Learning Priority


Epoch 4/50 - Duration: 1867.91s
Train Loss: 0.4921 - Val Loss: 1.3216
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9916, Val Acc: 0.9744
  Dominance: Train Acc: 0.9640, Val Acc: 0.8977
  Overall Val Accuracy: 0.9360
New best validation accuracy: 0.9360. Best model saved.

Epoch 5/50 - Phase 1: Learning Priority


...

Epoch 46/50 - Duration: 2284.36s
Train Loss: 0.4106 - Val Loss: 1.3725
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9994, Val Acc: 0.9736
  Dominance: Train Acc: 0.9990, Val Acc: 0.8776
  Overall Val Accuracy: 0.9256

Epoch 47/50 - Phase 2: Conserving Priority


Epoch 47/50 - Duration: 2282.65s
Train Loss: 0.4111 - Val Loss: 1.3779
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9996, Val Acc: 0.9741
  Dominance: Train Acc: 0.9986, Val Acc: 0.8726
  Overall Val Accuracy: 0.9234

Epoch 48/50 - Phase 2: Conserving Priority


Epoch 48/50 - Duration: 2303.75s
Train Loss: 0.4100 - Val Loss: 1.3734
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9996, Val Acc: 0.9711
  Dominance: Train Acc: 0.9990, Val Acc: 0.8790
  Overall Val Accuracy: 0.9251

Epoch 49/50 - Phase 2: Conserving Priority


Epoch 49/50 - Duration: 2309.86s
Train Loss: 0.4104 - Val Loss: 1.3530
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9994, Val Acc: 0.9719
  Dominance: Train Acc: 0.9991, Val Acc: 0.8893
  Overall Val Accuracy: 0.9306

Epoch 50/50 - Phase 2: Conserving Priority


Epoch 50/50 - Duration: 2274.42s
Train Loss: 0.4112 - Val Loss: 1.3651
  Occlusion: Train Acc: 0.0000, Val Acc: 0.0000
  Frame_Quality: Train Acc: 0.9989, Val Acc: 0.9736
  Dominance: Train Acc: 0.9990, Val Acc: 0.8840
  Overall Val Accuracy: 0.9288
Periodic checkpoint saved: E:\Morshedul\CoronarDominance\MTL_results\LCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results\periodic_checkpoints\checkpoint_epoch_50.pth
Saved last epoch model to E:\Morshedul\CoronarDominance\MTL_results\LCA\TwoPhasePriority_MTL_Densenet121_LSTM\training_results\TwoPhasePriority_Densenet121_temporal_mtl_last_epoch.pth (Epoch 50)
Two-phase priority temporal MTL training completed!


# INFERENCE

In [5]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                           accuracy_score, precision_recall_fscore_support,
                           precision_score, recall_score, f1_score)
from timeit import default_timer as timer
import re
from collections import defaultdict
import warnings
import json
warnings.filterwarnings('ignore')

In [6]:
class TaskSpecificDataset(Dataset):
    """Dataset that returns samples for a specific task only."""

    def __init__(self, root_dir, split_type, view_type, clip_length, task_name, transform=None, fold_num=1):
        self.root_dir = root_dir
        self.split_type = split_type
        self.view_type = view_type
        self.clip_length = clip_length
        self.task_name = task_name
        self.transform = transform
        self.fold_num = fold_num
        self.IGNORE_LABEL_VALUE = -1

        # Label mappings
        self.label_maps = {
            'occlusion': {'nonoccluded': 0, 'occluded': 1},
            'frame_quality': {'noninformative': 0, 'informative': 1},
            'dominance': {'rightdom': 0, 'leftdom': 1}
        }

        self.samples = []
        self._load_task_data()

    def _load_task_data(self):
        """Load data for the specific task."""
        if self.task_name == 'occlusion':
            self._load_occlusion_data()
        elif self.task_name == 'frame_quality':
            self._load_frame_quality_data()
        elif self.task_name == 'dominance':
            self._load_dominance_data()

    def _load_occlusion_data(self):
        """Load occlusion clips - ONE CLIP PER VIDEO."""
        if self.view_type != 'RCA':
            return

        csv_path = os.path.join(
            self.root_dir, 'occlusion', 'DATA_RCA', 'labels',
            f'occlusion_{self.split_type}_labels_fold_{self.fold_num}.csv'
        )

        if not os.path.exists(csv_path):
            return

        df = pd.read_csv(csv_path)
        video_frames = defaultdict(list)

        for _, row in df.iterrows():
            filename = str(row['filename'])
            label_str = str(row['label']).strip().lower()
            label = self.label_maps['occlusion'].get(label_str)

            if label is None:
                continue

            match = re.match(r'(.+)_frame_(\d+)\.png', filename)
            if match:
                video_id = match.group(1)
                frame_idx = int(match.group(2))
                path = os.path.join(self.root_dir, 'occlusion', 'DATA_RCA', label_str, filename)
                if os.path.exists(path):
                    video_frames[video_id].append((frame_idx, path, label, filename))

        print(f"Found {len(video_frames)} unique videos for occlusion task")

        # Create EXACTLY ONE clip per video
        for video_id, frames in video_frames.items():
            frames.sort(key=lambda x: x[0])  # Sort by frame index

            if len(frames) >= self.clip_length:
                # Strategy 1: Take middle frames (more representative)
                start_idx = (len(frames) - self.clip_length) // 2
                clip_frames = frames[start_idx:start_idx + self.clip_length]

                # Use the most common label in the clip (majority vote)
                labels_in_clip = [frame[2] for frame in clip_frames]
                video_label = max(set(labels_in_clip), key=labels_in_clip.count)

                self.samples.append({
                    'type': 'occlusion',
                    'video_id': video_id,
                    'frames': clip_frames,
                    'label': video_label,
                    'total_frames_in_video': len(frames)
                })
            else:
                # If video has fewer frames than clip_length, use all frames
                # and pad or skip based on your preference
                print(f"Warning: Video {video_id} has only {len(frames)} frames (< {self.clip_length})")
                # Option 1: Skip this video
                continue
            

        print(f"Created {len(self.samples)} occlusion clips (one per video)")

        # Verify we have the expected number
        if len(self.samples) != 107:
            print(f"WARNING: Expected 107 videos, but created {len(self.samples)} clips")
            print("This might be due to videos with insufficient frames or data filtering")

    def _load_frame_quality_data(self):
        """Load frame quality data."""
        csv_path = os.path.join(
            self.root_dir, 'framequality', f'DATA_{self.view_type}', 'labels',
            f'framequality_{self.split_type}_labels_fold_{self.fold_num}.csv'
        )

        if not os.path.exists(csv_path):
            return

        df = pd.read_csv(csv_path)

        for _, row in df.iterrows():
            filename = str(row['filename'])
            label_str = str(row['label']).strip().lower()
            label = self.label_maps['frame_quality'].get(label_str)

            if label is None:
                continue

            path = os.path.join(self.root_dir, 'framequality', f'DATA_{self.view_type}', label_str, filename)
            if os.path.exists(path):
                self.samples.append({
                    'type': 'frame_quality',
                    'path': path,
                    'label': label,
                    'filename': filename
                })

    def _load_dominance_data(self):
        """Load dominance data."""
        csv_path = os.path.join(
            self.root_dir, 'dominance', f'DATA_{self.view_type}', 'labels',
            f'dom_{self.split_type}_labels_fold_{self.fold_num}.csv'
        )

        if not os.path.exists(csv_path):
            return

        df = pd.read_csv(csv_path)

        for _, row in df.iterrows():
            filename = str(row['filename'])
            label_str = str(row['label']).strip().lower()
            label = self.label_maps['dominance'].get(label_str)

            if label is None:
                continue

            path = os.path.join(self.root_dir, 'dominance', f'DATA_{self.view_type}', label_str, filename)
            if os.path.exists(path):
                self.samples.append({
                    'type': 'dominance',
                    'path': path,
                    'label': label,
                    'filename': filename
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        if sample['type'] == 'occlusion':
            return self._get_occlusion_sample(sample)
        else:
            return self._get_frame_sample(sample)

    def _get_occlusion_sample(self, sample):
        """Get occlusion clip sample."""
        try:
            clip_frames = []
            for _, path, _, _ in sample['frames']:
                with Image.open(path) as img:
                    img = img.convert('L')
                    if self.transform:
                        img = self.transform(img)
                    clip_frames.append(img)

            clip_tensor = torch.stack(clip_frames)
            label = torch.tensor(sample['label'], dtype=torch.long)

            return {
                'task': 'occlusion',
                'input': clip_tensor,
                'label': label
            }
        except Exception as e:
            print(f"Error loading occlusion clip: {e}")
            return {
                'task': 'occlusion',
                'input': torch.zeros(self.clip_length, 1, 512, 512),
                'label': torch.tensor(self.IGNORE_LABEL_VALUE, dtype=torch.long)
            }

    def _get_frame_sample(self, sample):
        """Get single frame sample."""
        try:
            with Image.open(sample['path']) as img:
                img = img.convert('L')
                if self.transform:
                    img = self.transform(img)

            return {
                'task': sample['type'],
                'input': img,
                'label': torch.tensor(sample['label'], dtype=torch.long)
            }
        except Exception as e:
            print(f"Error loading frame: {e}")
            return {
                'task': sample['type'],
                'input': torch.zeros(1, 512, 512),
                'label': torch.tensor(self.IGNORE_LABEL_VALUE, dtype=torch.long)
            }


In [24]:
class InferenceMetrics:
    """Class to track and calculate inference metrics"""
    
    def __init__(self, task_name):
        self.task_name = task_name
        self.predictions = []
        self.true_labels = []
        self.inference_times = []
        self.gpu_memory_usage = []
        
    def update(self, preds, labels, inference_time, gpu_memory):
        """Update metrics with batch results"""
        if torch.is_tensor(preds):
            preds = preds.cpu().numpy()
        if torch.is_tensor(labels):
            labels = labels.cpu().numpy()
            
        self.predictions.extend(preds.flatten() if preds.ndim > 1 else preds)
        self.true_labels.extend(labels.flatten() if labels.ndim > 1 else labels)
        self.inference_times.append(inference_time)
        self.gpu_memory_usage.append(gpu_memory)
    
    def get_classification_report(self):
        """Generate classification report"""
        if len(self.predictions) == 0 or len(self.true_labels) == 0:
            return {}
        
        # Filter out ignore values (-1)
        valid_indices = [i for i, label in enumerate(self.true_labels) if label != -1]
        
        if len(valid_indices) == 0:
            return {}
            
        valid_preds = [self.predictions[i] for i in valid_indices]
        valid_labels = [self.true_labels[i] for i in valid_indices]
        
        # Get unique labels for target_names
        unique_labels = sorted(list(set(valid_labels)))
        
        if self.task_name == 'occlusion':
            target_names = ['nonoccluded', 'occluded']
        elif self.task_name == 'frame_quality':
            target_names = ['noninformative', 'informative']
        elif self.task_name == 'dominance':
            target_names = ['rightdom', 'leftdom']
        else:
            target_names = [f'class_{i}' for i in unique_labels]
        
        # Ensure target_names matches unique_labels
        target_names = [target_names[i] for i in unique_labels if i < len(target_names)]
        
        report = classification_report(
            valid_labels, valid_preds, 
            labels=unique_labels,
            target_names=target_names,
            output_dict=True,
            zero_division=0
        )
        return report
    
    def get_confusion_matrix(self):
        """Generate confusion matrix"""
        if len(self.predictions) == 0 or len(self.true_labels) == 0:
            return np.array([])
        
        # Filter out ignore values (-1)
        valid_indices = [i for i, label in enumerate(self.true_labels) if label != -1]
        
        if len(valid_indices) == 0:
            return np.array([])
            
        valid_preds = [self.predictions[i] for i in valid_indices]
        valid_labels = [self.true_labels[i] for i in valid_indices]
        
        unique_labels = sorted(list(set(valid_labels + valid_preds)))
        cm = confusion_matrix(valid_labels, valid_preds, labels=unique_labels)
        return cm
    
    def get_avg_inference_time(self):
        """Get average inference time per batch"""
        return np.mean(self.inference_times) if self.inference_times else 0.0
    
    def get_peak_gpu_memory(self):
        """Get peak GPU memory usage"""
        return max(self.gpu_memory_usage) if self.gpu_memory_usage else 0.0

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0.0

def create_inference_transform(img_size=(512, 512)):
    """Create transform for inference (no augmentation)"""
    return transforms.Compose([
        transforms.Resize(img_size, antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5485], std=[0.1407])
    ])

def load_model_from_checkpoint(checkpoint_path, backbone_type='resnet18', 
                              input_channels=1, hidden_dim=128, device='cuda'):
    """Load model from checkpoint"""
    
    # Create model
    model = create_optimized_model(
        backbone_type=backbone_type,
        input_channels=input_channels,
        hidden_dim=hidden_dim
    )
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    print(f"Model loaded from {checkpoint_path}")
    print(f"Checkpoint info - Epoch: {checkpoint.get('epoch', 'N/A')}, "
          f"Best Val Accuracy: {checkpoint.get('best_val_accuracy', 'N/A'):.4f}")
    
    return model

def run_task_inference(model, dataloader, task_name, device, ignore_label_value=-1):
    """Run inference for a specific task"""
    
    metrics = InferenceMetrics(task_name)
    model.eval()
    
    print(f"\n--- Running inference for {task_name.upper()} task ---")
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"Inferencing {task_name}")
        
        for batch_idx, batch_data in enumerate(pbar):
            batch_start_time = time.time()
            
            # Extract task-specific data
            task_type = batch_data['task']
            inputs = batch_data['input'].to(device, non_blocking=True)
            labels = batch_data['label'].to(device, non_blocking=True)
            
            # Prepare input dictionary based on task
            if task_name == 'occlusion':
                # For occlusion, input is already a clip (batch_size, clip_length, C, H, W)
                images_dict = {'occlusion_images': inputs}
                # Create dummy inputs for other tasks (not used in occlusion inference)
                dummy_single = torch.zeros(inputs.shape[0], 1, 1, inputs.shape[-2], inputs.shape[-1]).to(device)
                images_dict['frame_quality_images'] = dummy_single
                images_dict['dominance_images'] = dummy_single
                
            else:
                # For single frame tasks, add sequence dimension
                inputs_with_seq = inputs.unsqueeze(1)  # Add sequence dimension
                
                if task_name == 'frame_quality':
                    images_dict = {
                        'frame_quality_images': inputs_with_seq,
                        'occlusion_images': torch.zeros(inputs.shape[0], 5, inputs.shape[-3], inputs.shape[-2], inputs.shape[-1]).to(device),
                        'dominance_images': torch.zeros_like(inputs_with_seq).to(device)
                    }
                elif task_name == 'dominance':
                    images_dict = {
                        'dominance_images': inputs_with_seq,
                        'occlusion_images': torch.zeros(inputs.shape[0], 5, inputs.shape[-3], inputs.shape[-2], inputs.shape[-1]).to(device),
                        'frame_quality_images': torch.zeros_like(inputs_with_seq).to(device)
                    }
            
            # Forward pass
            outputs = model(images_dict)
            
            # Extract predictions for the current task
            """if task_name == 'occlusion':
                logits = outputs['occlusion'].view(-1, outputs['occlusion'].shape[-1])
                task_labels = labels.repeat(outputs['occlusion'].shape[1]).view(-1)  # Repeat for sequence length
            else:
                logits = outputs[task_name].squeeze(1)
                task_labels = labels"""

            if task_name == 'occlusion':
                # logits has shape [BatchSize, 5, 2] (B, clip_length, num_classes)
                logits = outputs['occlusion'] 
                
                # --- START FIX ---
                # Aggregate the 5 frame predictions into 1 video prediction
                # by averaging the logits across the clip (dim=1)
                video_logits = torch.mean(logits, dim=1)  # Shape is now [BatchSize, 2]
                
                # The original labels are already at the video level (shape [BatchSize])
                task_labels = labels 

                # Get the final prediction from the averaged logits
                _, predictions = torch.max(video_logits, 1) # Shape is [BatchSize]
                # --- END FIX ---

            else:
                logits = outputs[task_name].squeeze(1)
                task_labels = labels
                _, predictions = torch.max(logits, 1) # Add this line for consistency
            # Get predictions
            #_, predictions = torch.max(logits, 1)
            
            # Calculate batch inference time
            batch_inference_time = time.time() - batch_start_time
            
            # Get GPU memory usage
            gpu_memory = get_gpu_memory_usage()
            
            # Update metrics
            metrics.update(
                preds=predictions,
                labels=task_labels,
                inference_time=batch_inference_time,
                gpu_memory=gpu_memory
            )
            
            # Update progress bar
            pbar.set_postfix({
                'Batch Time': f'{batch_inference_time:.4f}s',
                'GPU Mem': f'{gpu_memory:.1f}MB'
            })
    
    return metrics

def save_confusion_matrix_plot(cm, task_name, output_dir, class_names=None):
    """Save confusion matrix plot"""
    if cm.size == 0:
        print(f"Warning: Empty confusion matrix for {task_name}")
        return
    
    plt.figure(figsize=(8, 6))
    
    # Set class names based on task
    if class_names is None:
        if task_name == 'occlusion':
            class_names = ['nonoccluded', 'occluded']
        elif task_name == 'frame_quality':
            class_names = ['noninformative', 'informative']
        elif task_name == 'dominance':
            class_names = ['rightdom', 'leftdom']
        else:
            class_names = [f'Class {i}' for i in range(cm.shape[0])]
    
    # Ensure class_names matches matrix dimensions
    if len(class_names) != cm.shape[0]:
        class_names = [f'Class {i}' for i in range(cm.shape[0])]
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {task_name.title()}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_dir, f'{task_name}_confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.close()

def run_complete_inference(checkpoint_path, data_root, view_type, fold_num=1,
                          backbone_type='resnet18', input_channels=1, hidden_dim=128,
                          batch_size=16, num_workers=4, clip_length=5,
                          output_dir='inference_results', device='cuda'):
    """Run complete inference pipeline for all tasks"""
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Starting inference pipeline...")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Data root: {data_root}")
    print(f"View type: {view_type}")
    print(f"Output directory: {output_dir}")
    
    # Load model
    model = load_model_from_checkpoint(
        checkpoint_path, backbone_type, input_channels, hidden_dim, device
    )
    
    # Create transform
    transform = create_inference_transform()
    
    # Define tasks based on view type
    if view_type == 'RCA':
        tasks = ['occlusion', 'frame_quality', 'dominance']
    else:  # LCA
        tasks = ['frame_quality', 'dominance']
    
    # Store results for all tasks
    all_results = {}
    
    for task_name in tasks:
        print(f"\n{'='*60}")
        print(f"PROCESSING TASK: {task_name.upper()}")
        print(f"{'='*60}")
        
        # Create dataset and dataloader for current task
        dataset = TaskSpecificDataset(
            root_dir=data_root,
            split_type='test',  # Change to 'val' or 'train' as needed
            view_type=view_type,
            clip_length=clip_length,
            task_name=task_name,
            transform=transform,
            fold_num=fold_num
        )
        
        if len(dataset) == 0:
            print(f"Warning: No data found for task {task_name}")
            continue
        
        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True,
            drop_last=False
        )
        
        print(f"Dataset size: {len(dataset)} samples")
        print(f"Number of batches: {len(dataloader)}")
        
        # Run inference
        metrics = run_task_inference(model, dataloader, task_name, device)
        
        # Generate results
        classification_rep = metrics.get_classification_report()
        confusion_mat = metrics.get_confusion_matrix()
        avg_inference_time = metrics.get_avg_inference_time()
        peak_gpu_memory = metrics.get_peak_gpu_memory()
        
        # Save confusion matrix plot
        if confusion_mat.size > 0:
            save_confusion_matrix_plot(confusion_mat, task_name, output_dir)
        
        # Store results
        task_results = {
            'task_name': task_name,
            'dataset_size': len(dataset),
            'avg_inference_time_per_batch': avg_inference_time,
            'peak_gpu_memory_mb': peak_gpu_memory,
            'classification_report': classification_rep,
            'confusion_matrix': confusion_mat.tolist() if confusion_mat.size > 0 else []
        }
        
        all_results[task_name] = task_results
        
        # Print summary
        print(f"\n--- {task_name.upper()} RESULTS SUMMARY ---")
        print(f"Dataset size: {len(dataset)} samples")
        print(f"Average inference time per batch: {avg_inference_time:.4f} seconds")
        print(f"Peak GPU memory usage: {peak_gpu_memory:.1f} MB")
        
        if classification_rep and 'accuracy' in classification_rep:
            print(f"Overall accuracy: {classification_rep['accuracy']:.4f}")
        
        if classification_rep and 'macro avg' in classification_rep:
            macro_avg = classification_rep['macro avg']
            print(f"Macro avg - Precision: {macro_avg['precision']:.4f}, "
                  f"Recall: {macro_avg['recall']:.4f}, F1: {macro_avg['f1-score']:.4f}")
        
        # Clean up GPU memory
        if device == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
    
    # Create comprehensive summary
    summary = {
        'inference_config': {
            'checkpoint_path': checkpoint_path,
            'data_root': data_root,
            'view_type': view_type,
            'fold_num': fold_num,
            'backbone_type': backbone_type,
            'input_channels': input_channels,
            'hidden_dim': hidden_dim,
            'batch_size': batch_size,
            'clip_length': clip_length,
            'device': device
        },
        'task_results': all_results,
        'overall_summary': {
            'total_tasks_processed': len(all_results),
            'tasks_processed': list(all_results.keys()),
            'total_inference_time': sum(result['avg_inference_time_per_batch'] * 
                                      (result['dataset_size'] // batch_size + 1)
                                      for result in all_results.values()),
            'max_gpu_memory_across_tasks': max(result['peak_gpu_memory_mb'] 
                                             for result in all_results.values()) if all_results else 0
        }
    }
    
    # Save results to JSON
    results_json_path = os.path.join(output_dir, 'inference_results.json')
    with open(results_json_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n{'='*60}")
    print("INFERENCE PIPELINE COMPLETED")
    print(f"{'='*60}")
    print(f"Results saved to: {output_dir}")
    print(f"JSON results: {results_json_path}")
    print(f"Confusion matrix plots saved for each task")
    
    # Print overall summary
    print(f"\nOVERALL SUMMARY:")
    print(f"Tasks processed: {summary['overall_summary']['tasks_processed']}")
    print(f"Total inference time: {summary['overall_summary']['total_inference_time']:.2f} seconds")
    print(f"Max GPU memory usage: {summary['overall_summary']['max_gpu_memory_across_tasks']:.1f} MB")
    
    return summary

In [25]:
config = {
        'checkpoint_path': r'E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\training_results\TwoPhasePriority_Resnet18_temporal_mtl_best.pth',
        'data_root': r'E:\Morshedul\CoronarDominance\MTL_DATASET',
        'view_type': 'RCA',  # or 'LCA'
        'fold_num': 1,
        'backbone_type': 'resnet18',
        'input_channels': 1,
        'hidden_dim': 128,
        'batch_size': 8,  # Adjust based on GPU memory
        'num_workers': 0,
        'clip_length': 5,
        'output_dir': r'E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\New_esting_results',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu'
    }
    
print("Starting MTL Model Inference...")
print(f"Device: {config['device']}")

# Run complete inference
results = run_complete_inference(**config)

print("\nInference completed successfully!")


Starting MTL Model Inference...
Device: cuda
Starting inference pipeline...
Checkpoint: E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\training_results\TwoPhasePriority_Resnet18_temporal_mtl_best.pth
Data root: E:\Morshedul\CoronarDominance\MTL_DATASET
View type: RCA
Output directory: E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\New_esting_results
Model loaded from E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\training_results\TwoPhasePriority_Resnet18_temporal_mtl_best.pth
Checkpoint info - Epoch: 9, Best Val Accuracy: 0.9081

PROCESSING TASK: OCCLUSION
Found 107 unique videos for occlusion task
Created 107 occlusion clips (one per video)
Dataset size: 107 samples
Number of batches: 14

--- Running inference for OCCLUSION task ---


Inferencing occlusion: 100%|█| 14/14 [00:04<00:00,  3.17it/s, Batch Time=0.0170s, GPU Mem



--- OCCLUSION RESULTS SUMMARY ---
Dataset size: 107 samples
Average inference time per batch: 0.0245 seconds
Peak GPU memory usage: 546.2 MB
Overall accuracy: 0.8131
Macro avg - Precision: 0.8255, Recall: 0.8014, F1: 0.8056

PROCESSING TASK: FRAME_QUALITY
Dataset size: 789 samples
Number of batches: 99

--- Running inference for FRAME_QUALITY task ---


Inferencing frame_quality: 100%|█| 99/99 [00:11<00:00,  8.39it/s, Batch Time=0.0230s, GPU



--- FRAME_QUALITY RESULTS SUMMARY ---
Dataset size: 789 samples
Average inference time per batch: 0.0269 seconds
Peak GPU memory usage: 554.2 MB
Overall accuracy: 0.9227
Macro avg - Precision: 0.9208, Recall: 0.9229, F1: 0.9218

PROCESSING TASK: DOMINANCE
Dataset size: 4149 samples
Number of batches: 519

--- Running inference for DOMINANCE task ---


Inferencing dominance: 100%|█| 519/519 [01:02<00:00,  8.33it/s, Batch Time=0.0220s, GPU M



--- DOMINANCE RESULTS SUMMARY ---
Dataset size: 4149 samples
Average inference time per batch: 0.0269 seconds
Peak GPU memory usage: 554.2 MB
Overall accuracy: 0.9704
Macro avg - Precision: 0.9206, Recall: 0.9339, F1: 0.9271

INFERENCE PIPELINE COMPLETED
Results saved to: E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\New_esting_results
JSON results: E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\New_esting_results\inference_results.json
Confusion matrix plots saved for each task

OVERALL SUMMARY:
Tasks processed: ['occlusion', 'frame_quality', 'dominance']
Total inference time: 16.99 seconds
Max GPU memory usage: 554.2 MB

Inference completed successfully!


In [22]:
# Create model
model = create_optimized_model(
    backbone_type='resnet18',
    input_channels=1,
    hidden_dim=128
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load checkpoint
checkpoint_path = r'E:\Morshedul\CoronarDominance\MTL_results\RCA\TwoPhasePriority_MTL_Resnet18_LSTM\training_results\TwoPhasePriority_Resnet18_temporal_mtl_best.pth'
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

XCAMultiTaskModel(
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True

In [18]:
checkpoint['model_state_dict'].keys()

odict_keys(['backbone.conv1.weight', 'backbone.bn1.weight', 'backbone.bn1.bias', 'backbone.bn1.running_mean', 'backbone.bn1.running_var', 'backbone.bn1.num_batches_tracked', 'backbone.layer1.0.conv1.weight', 'backbone.layer1.0.bn1.weight', 'backbone.layer1.0.bn1.bias', 'backbone.layer1.0.bn1.running_mean', 'backbone.layer1.0.bn1.running_var', 'backbone.layer1.0.bn1.num_batches_tracked', 'backbone.layer1.0.conv2.weight', 'backbone.layer1.0.bn2.weight', 'backbone.layer1.0.bn2.bias', 'backbone.layer1.0.bn2.running_mean', 'backbone.layer1.0.bn2.running_var', 'backbone.layer1.0.bn2.num_batches_tracked', 'backbone.layer1.1.conv1.weight', 'backbone.layer1.1.bn1.weight', 'backbone.layer1.1.bn1.bias', 'backbone.layer1.1.bn1.running_mean', 'backbone.layer1.1.bn1.running_var', 'backbone.layer1.1.bn1.num_batches_tracked', 'backbone.layer1.1.conv2.weight', 'backbone.layer1.1.bn2.weight', 'backbone.layer1.1.bn2.bias', 'backbone.layer1.1.bn2.running_mean', 'backbone.layer1.1.bn2.running_var', 'backbo